In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
%config InlineBackend.figure_format = 'retina' # configures the figure format for high-resolution displays.
%matplotlib inline
import matplotlib.pyplot as plt
import scipy.optimize as optimize # for root-finding functions
# Standard matplotlib color cycle
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## 2) Opening the dataset

This dataset has $18$ features (columns $1 \to 18$), and $500,000$ data points (rows). The $0^{th}$ column holds the labels 

In [3]:
csv_gz_file_path = 'data.csv'
df = pd.read_csv(csv_gz_file_path, index_col=None, header=None)
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,0.0,0.972861,0.653855,1.176225,1.157156,-1.739873,-0.874309,0.567765,-0.175000,0.810061,-0.252552,1.921887,0.889637,0.410772,1.145621,1.932632,0.994464,1.367815,0.040714
1,1.0,1.667973,0.064191,-1.225171,0.506102,-0.338939,1.672543,3.475464,-1.219136,0.012955,3.775174,1.045977,0.568051,0.481928,0.000000,0.448410,0.205356,1.321893,0.377584
2,1.0,0.444840,-0.134298,-0.709972,0.451719,-1.613871,-0.768661,1.219918,0.504026,1.831248,-0.431385,0.526283,0.941514,1.587535,2.024308,0.603498,1.562374,1.135454,0.180910
3,1.0,0.381256,-0.976145,0.693152,0.448959,0.891753,-0.677328,2.033060,1.533041,3.046260,-1.005285,0.569386,1.015211,1.582217,1.551914,0.761215,1.715464,1.492257,0.090719
4,1.0,1.309996,-0.690089,-0.676259,1.589283,-0.693326,0.622907,1.087562,-0.381742,0.589204,1.365479,1.179295,0.968218,0.728563,0.000000,1.083158,0.043429,1.154854,0.094859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,1.0,0.853325,-0.961783,-1.487277,0.678190,0.493580,1.647969,1.843867,0.276954,1.025105,-1.486535,0.892879,1.684429,1.674084,3.366298,1.046707,2.646649,1.389226,0.364599
4999996,0.0,0.951581,0.139370,1.436884,0.880440,-0.351948,-0.740852,0.290863,-0.732360,0.001360,0.257738,0.802871,0.545319,0.602730,0.002998,0.748959,0.401166,0.443471,0.239953
4999997,0.0,0.840389,1.419162,-1.218766,1.195631,1.695645,0.663756,0.490888,-0.509186,0.704289,0.045744,0.825015,0.723530,0.778236,0.752942,0.838953,0.614048,1.210595,0.026692
4999998,1.0,1.784218,-0.833565,-0.560091,0.953342,-0.688969,-1.428233,2.660703,-0.861344,2.116892,2.906151,1.232334,0.952444,0.685846,0.000000,0.781874,0.676003,1.197807,0.093689


## 3) Transforming labels so that they take value $\{−1, +1\}$

In [4]:
features = df.iloc[:, 1:]
X = features.copy()

labels=df.iloc[:, 0] # labels in the form {0,1}

y = labels.copy()
y *= 2
y -= 1               # y in the form {-1,+1}

In [36]:
def get_ridge_classifier_weights(X_train_scaled , y_train, a):
    # Create and fit the Ridge Classifier
    ridge_classifier = RidgeClassifier(alpha=a)
    ridge_classifier.fit(X_train_scaled , y_train)

    # Get the classifier's weights
    weights = ridge_classifier.coef_

    return weights


## 4-5) Splitting and processing the dataset

Implementing a function which first chooses the test-set then iteratively chooses train and validation sets, keeping the train set that gave the best accuracy on the validation set. The function then processes the train set and tests the model given as a parameter on the test set. <br>
We fit the scaler on the training set to ensure that the scaling parameters (mean and standard deviation) are based on the training data distribution. We don't fit it on the test or validation sets to maintain their independence and accurately evaluate how the model generalizes to new, unseen data (otherwise we would have overly optimistic model performance estimates and inaccurate assessments of how the model will perform on truly unseen data).

In [4]:
def train_model_with_early_stopping(X, y, model, epochs=5, patience=5):
    # Split the data into test and temporary data (20% for test, 80% for temporary)
    X_test, X_temp, y_test, y_temp = train_test_split(X, y, test_size=0.8, random_state=42)
    
    best_accuracy = 0
    best_weights = None
    no_improvement_count = 0

    for epoch in range(epochs):
        # Split the temporary data into train (60% of full dataset) and validation (20% of full dataset) in each epoch
        X_train, X_validation, y_train, y_validation = train_test_split(X_temp, y_temp, test_size=(0.2/0.8), train_size=(0.6/0.8))

        # Process the data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_validation_scaled = scaler.transform(X_validation)

        # Fit the model on the training data
        model.fit(X_train_scaled, y_train)

        # Make predictions on the validation data
        y_pred_validation = model.predict(X_validation_scaled)

        # Evaluate the model's performance on the validation set
        accuracy = accuracy_score(y_validation, y_pred_validation)
        #print(f"Epoch {epoch + 1} - Validation Accuracy: {accuracy}")

        # Check if the accuracy on the validation set improved
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_weights = model.coef_
            no_improvement_count = 0
        else:
            no_improvement_count += 1
            

        # Early stopping condition
        if no_improvement_count >= patience:
           # print(f"Early stopping at epoch {epoch + 1}")
            break

    # Use the best weights obtained during training
    model.coef_ = best_weights

    # Make predictions on the test data
    X_test_scaled = scaler.transform(X_test)
    y_pred = model.predict(X_test_scaled)

    # Evaluate the model's performance on the test set ONLY AT THE END AFTER TRAINING IS COMPLETE
    test_accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy on Test Set: {test_accuracy}")
    return model, test_accuracy

## 7) Running ridge classification with $\lambda = 0.1$

In [29]:
model1 = RidgeClassifier(alpha=0.1)
model1, test_accuracy1 = train_model_with_early_stopping(X, y, model1) # y contains -1,+1  

Accuracy on Test Set: 0.75795


## 8) Expression  for logistic regression 


Notation:<br>
$X = \begin{bmatrix}
X_1^{(1)} & \cdots & X_{d-1}^{(1)} & 1 \\
\vdots & \vdots & \vdots & \vdots \\
X_1^{(n)} & \cdots & X_{d-1}^{(n)} & 1
\end{bmatrix}$

$w^\star = \begin{bmatrix}
w_1^\star \\
\vdots \\
w_{d-1}^\star \\
w_0^\star
\end{bmatrix}$

The loss function is defined as $L(w) = \sum_{\mu=1}^n l(y_\mu, X_\mu\cdot x)$ where $l(y_\mu, X_\mu\cdot x) = -\ln{P(y_\mu | X_\mu \cdot w)}$.
In the notes, assuming the labels $y_\mu$ take values in $\{-1,+1\}$ we get the probability (p.27 eq 5.10) :
$$
P(y_\mu | X_\mu \cdot w)= \frac{e^{y_\mu\sum_{i=1}^d X_{\mu i}w_i}}{e^{\sum_{i=1}^d X_{\mu i}w_i}+e^{-\sum_{i=1}^d X_{\mu i}w_i}}
$$

If we consider that the labels $y_\mu$ take values in $\{0,+1\}$ we get the probability : 
$$
P(y_\mu | X_\mu \cdot w)= \frac{e^{y_\mu\sum_{i=1}^d X_{\mu i}w_i}}{e^{\sum_{i=1}^d X_{\mu i}w_i}+1}
$$

Therefore, the conditional probability that $y_\mu$ takes the value +1 is :
$\begin{align}
P(y_\mu=1| X_\mu \cdot w) &= \frac{e^{\sum_{i=1}^d X_{\mu i}w_i}}{e^{\sum_{i=1}^d X_{\mu i}w_i}+1} \\
                          &= \frac{1}{1+e^{-\sum_{i=1}^d X_{\mu i}w_i}}
\end{align}$  
which is the probability of the positive class that's found in the documentation. <br>

The maximum log-likelihood estimator is given by :
$\begin{align}
\hat{w}_{ML} &= \underset{w}{\mathrm{argmax}}\left[ \log{\prod_{\mu=1}^n}P(y_\mu | X_\mu \cdot w) \right] \\
\end{align}$



## 9) Running logistic regression for $\lambda=0.1$

In [30]:
model2 = LogisticRegression(C=(1/0.1), penalty='l2', solver='lbfgs', max_iter=100)
model2, test_accuracy2 = train_model_with_early_stopping(X, labels, model2) #labels contains 0,1

Accuracy on Test Set: 0.788165


## 10) Selecting the best value of $\lambda$

In [6]:
X_test, X_temp, y_test, y_temp = train_test_split(X, y, test_size=0.8, random_state=42) #for ridge we use y in {-1,+1}
X_train, X_validation, y_train, y_validation = train_test_split(X_temp, y_temp, test_size=(0.2/0.8), train_size=(0.6/0.8), random_state=42)

X_test, X_temp, label_test, label_temp = train_test_split(X, labels, test_size=0.8, random_state=42) #Logistic requires y in {0,+1}
X_train, X_validation, label_train, label_validation = train_test_split(X_temp, label_temp, test_size=(0.2/0.8), train_size=(0.6/0.8), random_state=42)

# Process the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

reg = np.logspace(-1, 4, 10)
Test_accuracies = np.empty((2, len(reg)))

for i, r in enumerate(reg):
    print(f"    Regularization = {r}")

    ridge = RidgeClassifier(alpha=r)
    logistic = LogisticRegression(C=1/r, penalty='l2', solver='lbfgs', max_iter=100)

    # Fit the modelS on the training data
    ridge.fit(X_train_scaled, y_train)
    logistic.fit(X_train_scaled, label_train)

    # Make predictions on the validation data
    y_pred_validation = ridge.predict(X_validation_scaled)
    # Evaluate the model's performance on the validation set
    Test_accuracies[0, i] = accuracy_score(y_validation, y_pred_validation)
    print(f"Ridge validation accuracy with regularisation {r} = {accuracy_score(y_validation, y_pred_validation)}")
 
    # Predictions and validation with the second model (Logistic)
    label_pred_validation = logistic.predict(X_validation_scaled)
    Test_accuracies[1, i] = accuracy_score(label_validation, label_pred_validation)
    print(f"Logistic validation accuracy with regularisation {r} = {accuracy_score(label_validation, label_pred_validation)}")
    print()

max_accuracy = np.max(Test_accuracies)
max_indices = np.unravel_index(np.argmax(Test_accuracies, axis=None), Test_accuracies.shape)

best_model_type = max_indices[0]  # 0 for Ridge, 1 for Logistic Regression
best_r_index = max_indices[1]  # Index of the best alpha value
best_r = reg[best_r_index]

print()
print(f"The best model type is {'Ridge' if best_model_type == 0 else 'Logistic Regression'}")
print(f"The best regularisation value is {best_r}")
print(f"The best validation accuracy is {max_accuracy}")

    Regularization = 0.1
Ridge validation accuracy with regularisation 0.1 = 0.756839
Logistic validation accuracy with regularisation 0.1 = 0.787666

    Regularization = 0.35938136638046275
Ridge validation accuracy with regularisation 0.35938136638046275 = 0.756839
Logistic validation accuracy with regularisation 0.35938136638046275 = 0.787669

    Regularization = 1.291549665014884
Ridge validation accuracy with regularisation 1.291549665014884 = 0.756841
Logistic validation accuracy with regularisation 1.291549665014884 = 0.787668

    Regularization = 4.641588833612779
Ridge validation accuracy with regularisation 4.641588833612779 = 0.756841
Logistic validation accuracy with regularisation 4.641588833612779 = 0.787679

    Regularization = 16.68100537200059
Ridge validation accuracy with regularisation 16.68100537200059 = 0.756839
Logistic validation accuracy with regularisation 16.68100537200059 = 0.787676

    Regularization = 59.94842503189409
Ridge validation accuracy with r

In [7]:
# Final test with the best model 
Finallogistic = LogisticRegression(C=1/best_r, penalty='l2', solver='lbfgs', max_iter=1000)
Finallogistic.fit(X_train_scaled, label_train)

X_test_scaled = scaler.transform(X_test)
label_pred = Finallogistic.predict(X_test_scaled)

# Evaluate the best model's performance on the test set ONLY AT THE END, AFTER TRAINING IS COMPLETE
final_test_accuracy = accuracy_score(label_test, label_pred)
print(f" --> Accuracy of best model on Test Set: {final_test_accuracy}")

 --> Accuracy of best model on Test Set: 0.788277


It is thus best to choose the Logistic regression with regularisation 774.3 giving an accuracy on the test-set of 0.788